# Path-dependent Option Pricing by Monte Carlo

### GBM Simulation

In [1]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

def simulate_paths(S, T, r, sigma, n_step, n_sim , seed=42):
    np.random.seed(seed)
    dt = T / n_step
    
    # Generate a matrix of Z ~ N(0, 1) with dimensions (nb of scenarios, nb of steps)
    Z = np.random.standard_normal((n_sim, n_step))
    growth = np.exp((r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z)
    
    # Paths matrix dimension (nb of scenarios, nb of steps + 1)
    paths = np.ones((n_sim, n_step + 1)) * S
    paths[:, 1:] = S * np.cumprod(growth, axis=1)
    
    return paths

### Parameters

In [2]:
S, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20
n_step, n_sim = 252, 100000

### European Option

In [3]:
def european_black_scholes(S, K, T, r, sigma, option_type='call'):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == 'call':
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
        
def european_mc(S, K, T, r, sigma, n_step, n_sim, option_type='call'):
    # Simulate
    paths = simulate_paths(S, T, r, sigma, n_step=252, n_sim=10000)
    
    # Compute payoffs
    sign = 1 if option_type == 'call' else -1
    payoff = np.maximum(sign * (paths[:, -1] - K), 0)

    # Option value
    price = np.exp(-r * T) * np.mean(payoff)
    
    # Standard Error: SE = std(payoffs) / sqrt(N)  [from CLT]
    se = np.exp(-r * T) * np.std(payoff, ddof=1) / np.sqrt(len(payoff))

    return price, se

In [4]:
# Pricing
bs_call = european_black_scholes(S, K, T, r, sigma, 'call')
bs_put  = european_black_scholes(S, K, T, r, sigma, 'put')

print(f"European Call (BS)           : {bs_call:.4f}")
print(f"European Put  (BS)           : {bs_put:.4f}")

European Call (BS)           : 10.4506
European Put  (BS)           : 5.5735


### Lookback Option

In [5]:
def lookback_mc(S, T, r, sigma, n_step, n_sim, option_type='call'):
    paths = simulate_paths(S, T, r, sigma, n_step, n_sim)
    S_T = paths[:, -1]

    if option_type == 'call':
        S_min  = paths.min(axis=1)   # Minimum Price
        payoffs = S_T - S_min
    else:
        S_max  = paths.max(axis=1)   # Maximum Price
        payoffs = S_max - S_T

    price = np.exp(-r * T) * np.mean(payoffs)
    se    = np.exp(-r * T) * np.std(payoffs, ddof=1) / np.sqrt(n_sim)

    return price, se

In [6]:
# Pricing
p_lc, se_lc = lookback_mc(S, T, r, sigma, n_step, n_sim, 'call')
p_lp, se_lp = lookback_mc(S, T, r, sigma, n_step, n_sim, 'put')

print(f"Lookback Call (Float Strike) : {p_lc:.4f} ± {1.96*se_lc:.4f}")
print(f"Lookback Put  (Float Strike) : {p_lp:.4f} ± {1.96*se_lp:.4f}")
print(f"European Call (BS)           : 10.4506")
print(f"European Put  (BS)           : 5.5735")

Lookback Call (Float Strike) : 16.5870 ± 0.0898
Lookback Put  (Float Strike) : 13.4291 ± 0.0611
European Call (BS)           : 10.4506
European Put  (BS)           : 5.5735


### Asian Option

In [7]:
def asian_mc(S, K, T, r, sigma, n_step, n_sim, option_type='call', asian_type='price'):
    paths = simulate_paths(S, T, r, sigma, n_step, n_sim)
    sign = 1 if option_type == 'call' else -1
    S_avg = paths[:, 1:].mean(axis=1)  # Exclude S0
    S_T   = paths[:, -1]

    if asian_type == 'price':
        payoffs = np.maximum(sign * (S_avg - K), 0)
    elif asian_type == 'strike':
        payoffs = np.maximum(sign * (S_T - S_avg), 0)
    
    price  = np.exp(-r * T) * payoffs.mean()
    se     = np.exp(-r * T) * payoffs.std(ddof=1) / np.sqrt(n_sim)
    
    return price, se

In [8]:
# Pricing
p_ap, se_ap = asian_mc(S, K, T, r, sigma, n_step, n_sim, 'call', 'price')
p_as, se_as = asian_mc(S, K, T, r, sigma, n_step, n_sim, 'call', 'strike')

print(f"Average Price Asian Call  : {p_ap:.4f} ± {1.96*se_ap:.4f}")
print(f"Average Strike Asian Call : {p_as:.4f} ± {1.96*se_as:.4f}")
print(f"European Call (BS)        : 10.4506")

Average Price Asian Call  : 5.7592 ± 0.0494
Average Strike Asian Call : 5.8279 ± 0.0524
European Call (BS)        : 10.4506


### Barrier Option

In [9]:
def barrier_mc(S, K, H, T, r, sigma, n_step, n_sim, option_type='call', barrier_type='down-and-out'):
    paths  = simulate_paths(S, T, r, sigma, n_step, n_sim)
    S_T    = paths[:, -1]
    sign   = 1 if option_type == 'call' else -1
    vanilla_payoffs = np.maximum(sign * (S_T - K), 0)

    if barrier_type == 'down-and-out':
        active = paths.min(axis=1) > H
    elif barrier_type == 'down-and-in':
        active = paths.min(axis=1) <= H
    elif barrier_type == 'up-and-out':
        active = paths.max(axis=1) < H
    elif barrier_type == 'up-and-in':
        active = paths.max(axis=1) >= H

    payoffs = vanilla_payoffs * active
    price   = np.exp(-r * T) * payoffs.mean()
    se      = np.exp(-r * T) * payoffs.std(ddof=1) / np.sqrt(n_sim)
    return price, se

In [10]:
# Pricing
H = 85.0
p_dao, se_dao = barrier_mc(S, K, H, T, r, sigma, n_step, n_sim, 'call', 'down-and-out')
p_dai, se_dai = barrier_mc(S, K, H, T, r, sigma, n_step, n_sim, 'call', 'down-and-in')
p_call, se_call = european_mc(S, K, T, r, sigma, n_step, n_sim, option_type='call')

print(f"Down-and-Out Call (H={H}) : {p_dao:.4f} ± {1.96*se_dao:.4f}")
print(f"Down-and-In  Call (H={H}) : {p_dai:.4f} ± {1.96*se_dai:.4f}")
print(f"Sum (D&O + D&I)           : {p_dao + p_dai:.4f}")
print(f"European Call (BS)        : 10.4506")
print(f"European Call (MC)        : {p_call:.4f} ± {1.96*se_call:.4f}")

Down-and-Out Call (H=85.0) : 9.9764 ± 0.0911
Down-and-In  Call (H=85.0) : 0.4229 ± 0.0163
Sum (D&O + D&I)           : 10.3994
European Call (BS)        : 10.4506
European Call (MC)        : 10.2211 ± 0.2834


In [11]:
for H in [75, 85, 95]:
    p, se = barrier_mc(S, K, H, T, r, sigma, n_step, n_sim, 'call', 'down-and-out')
    print(f"H = {H:3d} | D&O Call = {p:.4f} ± {1.96*se:.4f}")

H =  75 | D&O Call = 10.3896 ± 0.0908
H =  85 | D&O Call = 9.9764 ± 0.0911
H =  95 | D&O Call = 6.1564 ± 0.0834
